# AttnGAN Training and Inference NotebookThis notebook walks through the full AttnGAN workflow provided in the Paint4Poem repository.  It mirrors the original command line scripts while giving you a single interactive environment to:1. Prepare the dataset splits.2. Pre-train the DAMSM text encoder (optional but recommended).3. Train the AttnGAN model end-to-end.4. Run inference on custom prompts and visualise the generated artwork.> **Note:** Training AttnGAN from scratch is computationally expensive.  Expect to use a GPU with at least 11 GB of memory and allow many hours for convergence.  The cells below expose the exact entry points so that you can run shorter experiments or resume from checkpoints if desired.

## 0. PrerequisitesBefore running the notebook:- Download one of the Paint4Poem datasets and place it under `AttnGAN/../data`.  The sample subsets from the README are a good starting point for experimentation.- Ensure all Python dependencies are installed.  The project expects PyTorch, torchvision, nltk, pandas, Pillow, and other scientific Python packages.  When using a fresh environment run the `pip install` command in the cell below (uncomment the line first if needed).- If you intend to use CUDA, make sure the PyTorch version you install is compiled with GPU support and pass a valid GPU index in the configuration section later on.

In [ ]:
# Optional: install dependencies in a fresh environment.# !pip install torch torchvision nltk pandas pillow pyyaml python-dateutil tqdm

## 1. Project setupThe helper utilities in `AttnGAN/notebook_utils.py` wrap the original training scripts so the workflow can be driven from notebooks.  The next cell sets the working directory, ensures the AttnGAN sources are importable, and prints the paths being used.

In [ ]:
from pathlib import Pathimport osimport sys# Point the notebook to the AttnGAN project root.ATTNGAN_ROOT = Path.cwd()if ATTNGAN_ROOT.name != "AttnGAN":    ATTNGAN_ROOT = Path("..") / "AttnGAN"    os.chdir(ATTNGAN_ROOT)    ATTNGAN_ROOT = Path.cwd()if str(ATTNGAN_ROOT) not in sys.path:    sys.path.insert(0, str(ATTNGAN_ROOT))print(f"Working directory set to: {ATTNGAN_ROOT}")print(f"Python path now includes: {sys.path[0]}")

## 2. Prepare dataset splits

If you downloaded a raw dataset you can generate the training/test split files using `preprocess_data.py`. Update `DATA_DIR` to match the location of your dataset (for Kaggle datasets this might look like `/kaggle/input/tasets/poem-to-image`) and run the cell. The script will populate `train/filenames.pickle` and `test/filenames.pickle` in the dataset directory.


In [ ]:
DATA_DIR = "../data/Paint4Poem-Zikai-poem-subset/poem_image"
if not Path(DATA_DIR).exists():
    raise FileNotFoundError(f"DATA_DIR not found: {DATA_DIR}. Please download the dataset and update the path.")

print(f"Preparing splits for dataset at: {DATA_DIR}")
# Uncomment the line below to actually run the preprocessing step.
# !python preprocess_data.py --data_dir {DATA_DIR}


## 3. Configure and (optionally) pre-train DAMSMAttnGAN relies on a text encoder trained with DAMSM.  If you do not already have a pre-trained encoder, run the command below.  Replace the configuration path with the one that matches your dataset.

In [ ]:
DAMSM_CFG = "cfg/DAMSM/zikai_poem.yml"GPU_ID = 0  # Set to -1 to force CPU execution (very slow)print(f"Using DAMSM config: {DAMSM_CFG}")# Uncomment the line below to launch pre-training.  This step can take several hours.# !python pretrain_DAMSM.py --cfg {DAMSM_CFG} --gpu {GPU_ID}

## 4. Train AttnGANWith the DAMSM encoder ready, initialise the AttnGAN trainer.  The `prepare_trainer` helper loads the YAML config, seeds the random number generators, builds the dataloader, and returns a `condGANTrainer` instance.Update the `CFG_PATH`, `GPU_ID`, and `MANUAL_SEED` values as needed.  When you are ready to start training, run the final command in the cell (`run_training_loop(trainer)`).

In [ ]:
from notebook_utils import prepare_trainer, run_training_loopCFG_PATH = "cfg/zikai_poem_attn2_.yml"GPU_ID = 0  # Set to -1 for CPU-only trainingMANUAL_SEED = 1234trainer, train_dataset, train_loader, output_dir, used_seed = prepare_trainer(    cfg_path=CFG_PATH,    data_dir=DATA_DIR,    gpu_id=GPU_ID,    manual_seed=MANUAL_SEED,    split="train",)print(f"Trainer ready. Output directory: {output_dir}")print(f"Dataset size: {len(train_dataset)} images | Vocabulary size: {train_dataset.n_words}")print(f"Batches per epoch: {len(train_loader)} | Seed in use: {used_seed}")# Uncomment to start training.  Expect long runtimes for full training.# run_training_loop(trainer)

## 5. Load checkpoints for inferenceTo run inference you need paths to the trained generator (`netG`) and text encoder (`text_encoder`).  Update the variables below with the checkpoints you want to evaluate.  You can reuse the same dataset directory as during training, or point to a different evaluation split.Set `EVAL_CFG` to a configuration file where `TRAIN.FLAG` is `False` (see `cfg/eval_*.yml` for examples).

In [ ]:
from miscc.config import cfg

EVAL_CFG = "cfg/eval_poem_try.yml"
EVAL_GPU_ID = GPU_ID

eval_trainer, eval_dataset, eval_loader, eval_output_dir, _ = prepare_trainer(
    cfg_path=EVAL_CFG,
    data_dir=DATA_DIR,
    gpu_id=EVAL_GPU_ID,
    split="test",
)

print("Evaluation configuration loaded.")
print(f"Generator checkpoint: {cfg.TRAIN.NET_G}")
print(f"Text encoder checkpoint: {cfg.TRAIN.NET_E}")


## 6. Generate images from custom promptsUse `generate_from_prompts` to feed free-form text into the trained model.  The helper converts the prompts to token indices using the dataset vocabulary and calls `condGANTrainer.gen_example` under the hood.  The generated images are stored next to the generator checkpoint in subfolders named after each prompt.Run the cell below after updating the `prompts` list.  The output dictionary reports where the images were written.

In [ ]:
from notebook_utils import generate_from_prompts

prompts = [
    "A tranquil riverside pavilion surrounded by spring blossoms",
    "Mountains emerging from a morning mist with a single sailing boat",
]

results = generate_from_prompts(eval_trainer, prompts, eval_dataset.wordtoix)
for key, paths in results.items():
    print(f"{key}:")
    for path in paths:
        print(f"  - {path}")


## 7. Visualise generated artworkDisplay the generated images directly in the notebook.  If you produced multiple scales the helper will list each PNG file created for the prompt.

In [ ]:
from PIL import Imagefrom IPython.display import displayfor key, paths in results.items():    if not paths:        print(f"No images found for {key}. Check that generation completed successfully.")        continue    print(f"Prompt {key} ({prompts[int(key.split('_')[-1])]}):")    for image_path in paths:        display(Image.open(image_path))